## Silver Notebook

In [3]:
from pyspark.sql.types import *

orderSchema = StructType([
    StructField("OrderID", IntegerType()),
    StructField("CustomerID", StringType()),
    StructField("OrderDate", DateType()),
    StructField("Product", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("Price", FloatType())
])

df = spark.read.format("csv").option("header","true").schema(orderSchema).load("Files/Bronze/*.csv")

display(df)

StatementMeta(, 7dfd1f96-0c55-435d-8e79-47a6449187a4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10371c07-707d-428a-9cbb-79bad53cfa2a)

In [4]:
from pyspark.sql.functions import when, lit, col, current_timestamp, input_file_name

df = df.withColumn("FileName", input_file_name()) \
    .withColumn("IsFlagged", when(col("OrderDate") < '2019-06-01', True).otherwise(False)) \
    .withColumn("CreatedTS", current_timestamp()).withColumn("ModifiedTs", current_timestamp())

df = df.withColumn("CustomerID", when((col("CustomerID").isNull() | (col("CustomerID")=="")),lit("Unknown")).otherwise(col("CustomerID")))

StatementMeta(, 7dfd1f96-0c55-435d-8e79-47a6449187a4, 6, Finished, Available, Finished, False)

In [10]:
spark.sql("SELECT current_catalog(), current_schema()").show()

StatementMeta(, 999f7705-a1fe-4ccd-9e97-80e49f4aae52, 12, Finished, Available, Finished, False)

+-----------------+--------------------+
|current_catalog()|  current_database()|
+-----------------+--------------------+
|    spark_catalog|chimcobldhq2ah31e...|
+-----------------+--------------------+



In [11]:
from pyspark.sql.types import *
from delta.tables import *

DeltaTable.createIfNotExists(spark) \
    .tableName("sales_silver") \
    .addColumn("OrderID", IntegerType()) \
    .addColumn("CustomerID", StringType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("Product", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("Price", FloatType()) \
    .addColumn("FileName", StringType()) \
    .addColumn("IsFlagged", BooleanType()) \
    .addColumn("CreatedTS", DateType()) \
    .addColumn("ModifiedTS", StringType()) \
    .execute()



StatementMeta(, 999f7705-a1fe-4ccd-9e97-80e49f4aae52, 13, Finished, Available, Finished, False)

In [5]:
from delta.tables import *

# deltaTable = DeltaTable.forPath(spark,'Tables/sales_silver')
path = "abfss://DataEngineering@onelake.dfs.fabric.microsoft.com/Sales.Lakehouse/Tables/dbo/sales_silver"
deltaTable = DeltaTable.forName(spark, "sales_silver")

dfUpdates = df

deltaTable.alias('silver') \
.merge(
    dfUpdates.alias('updates'),
    'silver.Quantity = updates.Quantity and silver.OrderDate = updates.OrderDate'
) \
.whenMatchedUpdate( set =
{

}) \
.whenNotMatchedInsert(values =
{

}) \
.execute()

StatementMeta(, 7dfd1f96-0c55-435d-8e79-47a6449187a4, 7, Finished, Available, Finished, False)